In [0]:
#Author : Aimun

# ERP Bronze → Silver — Latest-Version Fix

**Updated rule:** after exact deduplication, parse `last_updated` as TIMESTAMP and keep only the latest row per `inventory_id` using `row_number()`.

**Target:** 518 Bronze → 508 after exact dedup → **500 final Silver rows**.


# Pharma Manufacturing Project — ERP Bronze → Silver

**Step-by-step Databricks notebook**

This notebook follows the same Bronze → Silver principles used for Historian.

Project rules:
- Read Bronze as strings first.
- Remove **exact duplicates** before cleaning.
- Do **not** automatically delete rows just because an `inventory_id` appears more than once; flag them for review.
- Standardise `site` to title case and `product_code` to upper case.
- Convert comma-formatted quantities safely.
- Parse dates/timestamps safely using tolerant parsing.
- Blank `expected_delivery_date` values are allowed and are **not** automatically treated as data-quality errors.
- Create `reorder_flag` when `available_qty < reorder_level`.
- Create `stock_gap` for materials below reorder level.
- Write Silver as Delta in overwrite mode for this training/full-refresh pipeline.


In [0]:
# STEP 1 - Import PySpark functions

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# STEP 2 - Define Bronze, Silver and Snowflake export paths

bronze_path = "abfss://bronze@pharmadatalakeuk.dfs.core.windows.net/erp/erp_inventory.csv"

silver_path = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net/erp"

snowflake_export_path = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net/export/erp"

spark.conf.set("spark.sql.session.timeZone", "UTC")


In [0]:
# STEP 3 - Read a FRESH Bronze ERP DataFrame
# Everything stays as STRING initially.

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(bronze_path)
)

display(df_raw.limit(10))

In [0]:
# STEP 4 - Record Bronze row count and inspect schema

bronze_count = df_raw.count()

print("Bronze row count:", bronze_count)

df_raw.printSchema()

In [0]:
# STEP 5 - Remove exact duplicate rows BEFORE cleaning

df = df_raw.dropDuplicates()

dedup_count = df.count()

print("Rows before deduplication:", bronze_count)
print("Rows after deduplication :", dedup_count)
print("Exact duplicates removed :", bronze_count - dedup_count)

In [0]:
# STEP 6 - Standardise identifier columns

df = (
    df
    .withColumn(
        "inventory_id",
        F.upper(F.trim(F.col("inventory_id")))
    )
    .withColumn(
        "material_id",
        F.upper(F.trim(F.col("material_id")))
    )
    .withColumn(
        "product_code",
        F.upper(F.trim(F.col("product_code")))
    )
)

In [0]:
# STEP 7 - Standardise text columns
#
# site follows the shared project convention:
# Dublin, Houston, Singapore, Wuxi

df = (
    df
    .withColumn(
        "site",
        F.initcap(F.lower(F.trim(F.col("site"))))
    )
    .withColumn(
        "material_type",
        F.upper(F.trim(F.col("material_type")))
    )
    .withColumn(
        "material_name",
        F.trim(F.col("material_name"))
    )
    .withColumn(
        "supplier",
        F.upper(F.trim(F.col("supplier")))
    )
    .withColumn(
        "uom",
        F.lower(F.trim(F.col("uom")))
    )
)

In [0]:
# STEP 8 - Inspect standardised categorical values

print("Sites")
display(
    df.groupBy("site")
      .count()
      .orderBy("site")
)

print("Material types")
display(
    df.groupBy("material_type")
      .count()
      .orderBy("material_type")
)

print("Units of measure")
display(
    df.groupBy("uom")
      .count()
      .orderBy("uom")
)

In [0]:
# STEP 9 - Keep raw copies of fields that will be converted

df = (
    df
    .withColumn("available_qty_raw", F.col("available_qty"))
    .withColumn("reorder_level_raw", F.col("reorder_level"))
    .withColumn("expected_delivery_date_raw", F.col("expected_delivery_date"))
    .withColumn("last_updated_raw", F.col("last_updated"))
)

In [0]:
# STEP 10 - KEEP ONLY THE LATEST VERSION OF EACH inventory_id
#
# WHY:
# - STEP 5 removed 10 exact duplicate rows: 518 -> 508.
# - 8 inventory IDs still have an older and a newer version.
# - Those rows are NOT exact duplicates because quantity and/or last_updated differ.
# - ERP Silver must contain one CURRENT row per inventory_id.
#
# IMPORTANT:
# Parse last_updated as a real TIMESTAMP BEFORE sorting.
# 20 rows use dd/MM/yyyy HH:mm, so sorting the raw text can choose the wrong row.

# 10A - Count how many source versions each inventory_id has
inventory_id_window = Window.partitionBy("inventory_id")

df = df.withColumn(
    "inventory_id_record_count",
    F.count(F.lit(1)).over(inventory_id_window)
)

df = df.withColumn(
    "inventory_id_duplicate_flag",
    F.col("inventory_id_record_count") > 1
)

# 10B - Classify last_updated format
df = df.withColumn(
    "last_updated_format",
    F.when(
        F.trim(F.col("last_updated_raw")).rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"),
        F.lit("yyyy-MM-dd HH:mm:ss")
    )
    .when(
        F.trim(F.col("last_updated_raw")).rlike(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$"),
        F.lit("dd/MM/yyyy HH:mm")
    )
    .otherwise(F.lit("unknown"))
)

# Check the timestamp formats BEFORE version filtering.
print("last_updated formats before latest-version filtering")
display(df.groupBy("last_updated_format").count().orderBy("last_updated_format"))

# 10C - Parse last_updated safely into a TIMESTAMP
last_updated_iso = F.expr(
    "try_to_timestamp(trim(last_updated_raw), 'yyyy-MM-dd HH:mm:ss')"
)

last_updated_dmy = F.expr(
    "try_to_timestamp(trim(last_updated_raw), 'dd/MM/yyyy HH:mm')"
)

df = df.withColumn(
    "last_updated",
    F.coalesce(last_updated_iso, last_updated_dmy)
)

df = df.withColumn(
    "last_updated_invalid_flag",
    F.col("last_updated").isNull()
)

# 10D - Rank versions newest -> oldest within each inventory_id
latest_inventory_window = (
    Window.partitionBy("inventory_id")
          .orderBy(F.col("last_updated").desc_nulls_last())
)

df = df.withColumn(
    "_latest_row_number",
    F.row_number().over(latest_inventory_window)
)

# 10E - Keep only row number 1 (the latest version)
df = (
    df
    .filter(F.col("_latest_row_number") == 1)
    .drop("_latest_row_number")
)

# 10F - Validate the new ERP Silver grain
print("Rows after keeping latest inventory version:", df.count())
print("Unique inventory IDs:", df.select("inventory_id").distinct().count())

duplicate_ids_remaining = (
    df.groupBy("inventory_id")
      .count()
      .filter(F.col("count") > 1)
)

print("Duplicate inventory IDs remaining:", duplicate_ids_remaining.count())
display(duplicate_ids_remaining)


In [0]:
# STEP 11 - Profile comma-formatted available quantities

df = df.withColumn(
    "available_qty_had_comma",
    F.coalesce(
        F.col("available_qty_raw").contains(","),
        F.lit(False)
    )
)

print(
    "Rows with comma-formatted available_qty:",
    df.filter(F.col("available_qty_had_comma") == True).count()
)

In [0]:
# STEP 12 - Clean commas/spaces from available_qty before parsing

df = df.withColumn(
    "available_qty_clean_text",
    F.regexp_replace(
        F.trim(F.col("available_qty_raw")),
        ",",
        ""
    )
)

In [0]:
# STEP 13 - Parse available_qty and reorder_level SAFELY
#
# try_cast returns NULL instead of failing on malformed input.

df = (
    df
    .withColumn(
        "available_qty",
        F.expr("try_cast(available_qty_clean_text AS DOUBLE)")
    )
    .withColumn(
        "reorder_level",
        F.expr("try_cast(trim(reorder_level_raw) AS DOUBLE)")
    )
)

In [0]:
# STEP 14 - Create numeric data-quality flags

df = (
    df
    .withColumn(
        "available_qty_missing_flag",
        F.col("available_qty_raw").isNull()
        | (F.trim(F.col("available_qty_raw")) == "")
    )
    .withColumn(
        "available_qty_invalid_numeric",
        (
            F.col("available_qty_raw").isNotNull()
            & (F.trim(F.col("available_qty_raw")) != "")
            & F.col("available_qty").isNull()
        )
    )
    .withColumn(
        "negative_available_qty_flag",
        F.coalesce(
            F.col("available_qty") < 0,
            F.lit(False)
        )
    )
    .withColumn(
        "reorder_level_missing_flag",
        F.col("reorder_level_raw").isNull()
        | (F.trim(F.col("reorder_level_raw")) == "")
    )
    .withColumn(
        "reorder_level_invalid_numeric",
        (
            F.col("reorder_level_raw").isNotNull()
            & (F.trim(F.col("reorder_level_raw")) != "")
            & F.col("reorder_level").isNull()
        )
    )
)

In [0]:
# STEP 15 - Flag missing product_code
#
# product_code is important because ERP may later relate to MES
# using site + product_code.

df = df.withColumn(
    "product_code_missing_flag",
    F.col("product_code").isNull()
    | (F.trim(F.col("product_code")) == "")
)

In [0]:
# STEP 16 - Classify expected_delivery_date source formats
#
# Blank delivery dates can be genuine and are kept as blank.

df = df.withColumn(
    "expected_delivery_date_format",

    F.when(
        F.col("expected_delivery_date_raw").isNull()
        | (F.trim(F.col("expected_delivery_date_raw")) == ""),
        F.lit("blank")
    )
    .when(
        F.trim(F.col("expected_delivery_date_raw")).rlike(
            r"^\d{4}-\d{2}-\d{2}$"
        ),
        F.lit("yyyy-MM-dd")
    )
    .when(
        F.trim(F.col("expected_delivery_date_raw")).rlike(
            r"^\d{2}/\d{2}/\d{4}$"
        ),
        F.lit("dd/MM/yyyy")
    )
    .when(
        F.trim(F.col("expected_delivery_date_raw")).rlike(
            r"^\d{2}-[A-Za-z]{3}-\d{4}$"
        ),
        F.lit("dd-MMM-yyyy")
    )
    .otherwise(F.lit("unknown"))
)

display(
    df.groupBy("expected_delivery_date_format")
      .count()
      .orderBy("expected_delivery_date_format")
)

In [0]:
# STEP 17 - Parse expected_delivery_date SAFELY
#
# We support all formats present in the source:
# 1. yyyy-MM-dd
# 2. dd/MM/yyyy
# 3. dd-MMM-yyyy

expected_date_iso = F.expr(
    "try_to_timestamp(trim(expected_delivery_date_raw), 'yyyy-MM-dd')"
)

expected_date_dmy = F.expr(
    "try_to_timestamp(trim(expected_delivery_date_raw), 'dd/MM/yyyy')"
)

expected_date_text_month = F.expr(
    "try_to_timestamp(trim(expected_delivery_date_raw), 'dd-MMM-yyyy')"
)

df = df.withColumn(
    "expected_delivery_date",
    F.to_date(
        F.coalesce(
            expected_date_iso,
            expected_date_dmy,
            expected_date_text_month
        )
    )
)

In [0]:
# STEP 18 - Create expected delivery date flags
#
# A blank expected delivery date is NOT automatically a DQ problem.

df = (
    df
    .withColumn(
        "expected_delivery_date_missing_flag",
        F.col("expected_delivery_date_raw").isNull()
        | (F.trim(F.col("expected_delivery_date_raw")) == "")
    )
    .withColumn(
        "expected_delivery_date_invalid_flag",
        (
            F.col("expected_delivery_date_raw").isNotNull()
            & (F.trim(F.col("expected_delivery_date_raw")) != "")
            & F.col("expected_delivery_date").isNull()
        )
    )
)

In [0]:
# STEP 22 - Validate date and timestamp parsing

display(
    df.select(
        "inventory_id",
        "expected_delivery_date_raw",
        "expected_delivery_date_format",
        "expected_delivery_date",
        "expected_delivery_date_missing_flag",
        "expected_delivery_date_invalid_flag",
        "last_updated_raw",
        "last_updated_format",
        "last_updated",
        "last_updated_invalid_flag"
    ).limit(50)
)

In [0]:
# STEP 23 - Feature engineering: create reorder_flag
#
# True  = available_qty is below reorder_level
# False = stock is at/above reorder_level
# NULL  = cannot determine because one of the required values is missing

df = df.withColumn(
    "reorder_flag",

    F.when(
        F.col("available_qty").isNull()
        | F.col("reorder_level").isNull(),
        F.lit(None).cast("boolean")
    )
    .otherwise(
        F.col("available_qty") < F.col("reorder_level")
    )
)

In [0]:
# STEP 24 - Feature engineering: create stock_gap
#
# For a reorder row:
# stock_gap = reorder_level - available_qty
#
# If stock is not below reorder level, gap = 0.
# If quantity/reorder level is unknown, gap stays NULL.

df = df.withColumn(
    "stock_gap",

    F.when(
        F.col("available_qty").isNull()
        | F.col("reorder_level").isNull(),
        F.lit(None).cast("double")
    )
    .when(
        F.col("available_qty") < F.col("reorder_level"),
        F.col("reorder_level") - F.col("available_qty")
    )
    .otherwise(F.lit(0.0))
)

In [0]:
# STEP 25 - Feature engineering: create inventory_status

df = df.withColumn(
    "inventory_status",

    F.when(
        F.col("available_qty").isNull()
        | F.col("reorder_level").isNull(),
        F.lit("UNKNOWN")
    )
    .when(
        F.col("available_qty") < F.col("reorder_level"),
        F.lit("REORDER")
    )
    .otherwise(F.lit("OK"))
)

In [0]:
# STEP 26 - Create the overall ERP data-quality flag
#
# IMPORTANT:
# expected_delivery_date_missing_flag is NOT included because a blank
# delivery date can be genuine for well-stocked materials.

df = df.withColumn(
    "data_quality_issue",

    F.col("inventory_id_duplicate_flag")
    | F.col("available_qty_missing_flag")
    | F.col("available_qty_invalid_numeric")
    | F.col("negative_available_qty_flag")
    | F.col("reorder_level_missing_flag")
    | F.col("reorder_level_invalid_numeric")
    | F.col("product_code_missing_flag")
    | F.col("expected_delivery_date_invalid_flag")
    | F.col("last_updated_invalid_flag")
)

In [0]:
# STEP 27 - Preview rows that need reorder

display(
    df.filter(F.col("reorder_flag") == True)
      .select(
          "inventory_id",
          "material_id",
          "material_name",
          "material_type",
          "product_code",
          "site",
          "available_qty",
          "uom",
          "reorder_level",
          "stock_gap",
          "expected_delivery_date",
          "supplier",
          "inventory_status"
      )
      .orderBy(F.desc("stock_gap"))
      .limit(100)
)

In [0]:
# STEP 28 - Preview rows with data-quality issues

display(
    df.filter(F.col("data_quality_issue") == True)
      .select(
          "inventory_id",
          "material_id",
          "product_code",
          "site",
          "available_qty",
          "reorder_level",
          "inventory_id_duplicate_flag",
          "available_qty_missing_flag",
          "available_qty_invalid_numeric",
          "negative_available_qty_flag",
          "reorder_level_missing_flag",
          "product_code_missing_flag",
          "expected_delivery_date_invalid_flag",
          "last_updated_invalid_flag",
          "data_quality_issue"
      )
      .orderBy("inventory_id")
      .limit(100)
)

In [0]:
# STEP 29 - Build the final Silver DataFrame
#
# Bronze keeps the original raw values.
# Temporary working columns are removed from final Silver.

df_silver = df.drop(
    "available_qty_raw",
    "available_qty_clean_text",
    "reorder_level_raw",
    "expected_delivery_date_raw",
    "last_updated_raw"
)

In [0]:
# FINAL VALIDATION - ERP Silver must now be one row per inventory_id

silver_count = df_silver.count()

duplicate_ids_remaining = (
    df_silver.groupBy("inventory_id")
             .count()
             .filter(F.col("count") > 1)
             .count()
)

print("========================================")
print("ERP SILVER FINAL VALIDATION")
print("========================================")
print("Bronze rows                         :", bronze_count)
print("After exact deduplication           :", dedup_count)
print("Exact duplicates removed            :", bronze_count - dedup_count)
print("Older versions removed              :", dedup_count - silver_count)
print("FINAL SILVER ROWS                   :", silver_count)
print("UNIQUE INVENTORY IDs                :", df_silver.select("inventory_id").distinct().count())
print("DUPLICATE INVENTORY IDs REMAINING   :", duplicate_ids_remaining)

assert silver_count == 500, f"Expected 500 ERP Silver rows, got {silver_count}"
assert df_silver.select("inventory_id").distinct().count() == 500, "Expected 500 unique inventory IDs"
assert duplicate_ids_remaining == 0, "Duplicate inventory IDs still remain"

print("ERP Silver latest-version rule PASSED.")


In [0]:
# STEP 31 - Inspect final ERP Silver schema

df_silver.printSchema()

In [0]:
# STEP 32 - Preview final ERP Silver data

display(
    df_silver.select(
        "inventory_id",
        "material_id",
        "material_name",
        "material_type",
        "product_code",
        "site",
        "available_qty",
        "uom",
        "reorder_level",
        "expected_delivery_date",
        "supplier",
        "last_updated",
        "inventory_id_duplicate_flag",
        "available_qty_had_comma",
        "negative_available_qty_flag",
        "expected_delivery_date_missing_flag",
        "reorder_flag",
        "stock_gap",
        "inventory_status",
        "data_quality_issue"
    ).limit(100)
)

In [0]:
# STEP 33 - Write ERP Silver as Delta

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path)
)

print("ERP Silver write completed.")

In [0]:
# STEP 34 - Read ERP Silver Delta back to verify the write

df_check = (
    spark.read
    .format("delta")
    .load(silver_path)
)

print("Rows read back from ERP Silver:", df_check.count())

display(df_check.limit(10))

In [0]:
# STEP 35 - Export the current ERP Silver snapshot as Parquet for Snowflake
#
# We export from the Delta read-back DataFrame so Snowflake receives only the
# current valid snapshot. Overwrite keeps this export folder clean on reruns.

(
    df_check.write
    .mode("overwrite")
    .parquet(snowflake_export_path)
)

print("ERP Parquet export completed.")
print("Snowflake export path:", snowflake_export_path)


In [0]:
# STEP 36 - Final completion message

print("========================================")
print("ERP BRONZE -> SILVER SUCCESSFUL")
print("========================================")
print("Silver Delta path:", silver_path)
print("Snowflake Parquet export path:", snowflake_export_path)


## ERP latest-version fix applied

The notebook now applies the rule in executable code **before the final Silver DataFrame is written**:

1. Remove 10 exact duplicates: **518 → 508**.
2. Parse `last_updated` as a real timestamp, including `dd/MM/yyyy HH:mm` rows.
3. Use `row_number()` partitioned by `inventory_id`, ordered by `last_updated DESC`.
4. Keep row number 1 only.
5. Final ERP Silver grain: **500 rows / 500 unique inventory IDs / 0 duplicate IDs**.
6. Overwrite the Delta Silver table and the Parquet export used by Snowflake.

**Important:** the numbers above describe the rule. The executable FINAL VALIDATION cell contains assertions and will fail if the actual result is not 500/500/0.

### Snowflake reload after the notebook succeeds

```sql
TRUNCATE TABLE PHARMA.SILVER.ERP;

COPY INTO PHARMA.SILVER.ERP
FROM @PHARMA.SILVER.ADLS_SILVER_STAGE/export/erp/
FILE_FORMAT = (FORMAT_NAME = 'PARQUET_FF')
PATTERN = '.*part-.*[.]parquet'
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
FORCE = TRUE;

SELECT COUNT(*) FROM PHARMA.SILVER.ERP;

SELECT inventory_id, COUNT(*)
FROM PHARMA.SILVER.ERP
GROUP BY inventory_id
HAVING COUNT(*) > 1;
```

Expected Snowflake result: **500 rows** and **no duplicate inventory IDs**.
